

1.Successfully engineered price sensitivity and customer tenure features. The data is now ready for modeling."



In [3]:
import pandas as pd
import numpy as np

# 1. Load the datasets with exact filenames shown in your sidebar
df_client = pd.read_csv('clean_data_after_eda.csv')
df_price = pd.read_csv('price_data (1).csv')

print(" Files loaded successfully!")

# 2. Calculate Price Sensitivity (Difference between Dec and Jan)
# Ensure price_date is in datetime format
df_price['price_date'] = pd.to_datetime(df_price['price_date'])

# Group by client ID and extract the first (Jan) and last (Dec) prices
price_features = df_price.sort_values(by=['id', 'price_date']).groupby('id').agg({
    'price_off_peak_var': ['first', 'last']
}).reset_index()

# Rename columns to be more professional
price_features.columns = ['id', 'price_jan_var', 'price_dec_var']

# Create the new sensitivity feature
price_features['offpeak_diff_dec_jan'] = price_features['price_dec_var'] - price_features['price_jan_var']

# 3. Merge this new feature into the main client dataframe
df_final = pd.merge(df_client, price_features[['id', 'offpeak_diff_dec_jan']], on='id', how='left')

# 4. Create Tenure Feature (Years as a customer)
df_final['date_activ'] = pd.to_datetime(df_final['date_activ'])
df_final['date_end'] = pd.to_datetime(df_final['date_end'])
df_final['tenure_years'] = ((df_final['date_end'] - df_final['date_activ']).dt.days / 365).round(2)

# 5. Exporting for Task 5 (Machine Learning)
df_final.to_csv('final_data_for_modeling.csv', index=False)

print(" SUCCESS: Features engineered and file saved!")
df_final[['id', 'offpeak_diff_dec_jan', 'tenure_years']].head()

 Files loaded successfully!
 SUCCESS: Features engineered and file saved!


,id,offpeak_diff_dec_jan,tenure_years
0,24011ae4ebbe3035111d65fa7c15bc57,0.020057,3.00
1,d29c2c54acc38ff3c0614d0a653813dd,-0.003767,7.03
2,764c75f661154dac3a6c254cd082ea7d,-0.004670,6.01
3,bba03439a292a1e166f80264c16191cb,-0.004547,6.01
4,149d57cf92fc41cf94415803a877cb4b,-0.006192,6.15


In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Prepare data for Machine Learning
# We only take numeric columns to avoid "string to float" errors
X = df_final.select_dtypes(include=['number']).drop(columns=['churn'], errors='ignore')
y = df_final['churn']

# Fill any missing values with 0
X = X.fillna(0)

# 2. Split into Training and Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Final Accuracy and Report
prediction = model.predict(X_test)

print(f"✅ Model Training Complete!")
print(f"Accuracy Score: {accuracy_score(y_test, prediction):.4f}")
print("\n--- Classification Report ---")
print(classification_report(y_test, prediction))

✅ Model Training Complete!
Accuracy Score: 0.8984

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      2617
           1       0.70      0.05      0.09       305

    accuracy                           0.90      2922
   macro avg       0.80      0.52      0.52      2922
weighted avg       0.88      0.90      0.86      2922

